# 多会话交接

会话结束了，任务还没有。交接包是将“agent跑了一个小时”变成“下一个会话在第一分钟就准备就绪”的物料。有意识地去构建它，而不是事后才想起来。

## 问题描述

会话结束了，agent说“太棒了，我们取得了进展”。打开下一个会话，下一个agent问“我们该从哪里开始？”。第一个agent的答案丢了。下一个agent需要重新发现、跑相同的命令、然后问相同的问题，花30分钟来重新发现上个会话最后30秒的东西。

一个坏的交接会让任务过程中的每个会话都付出代价。修复方式是会话结束时自动生成的一个包：改了什么、为什么、尝试了哪些、哪些失败了、哪些还没做、下一步做什么。

## 基本概念
```mermaid
flowchart TD
A[agent_state.json]-->E[generate_handoff.py]
B[verification_report.json]-->E
C[review_report.json]-->E
D[feedback_record.jsonl]-->E
E-->F[handoff.md + handoff.json]
F-->G[Next Session]
```

### 每次交接携带的七个信息

|字段|回答了什么|
|---|---|
|summary|做了什么的一段话|
|changed_files|差异概览|
|command_runs|跑了哪些命令|
|failed_attempts|做了哪些尝试、为什么失败|
|open_risks|严重性上来说有哪些坑|
|next_action|下一个会话需要采取的第一个具体步骤|
|verdict_pointer|指向验证+审查报告的路径|

下一个任务字段是承重的。没有这个字段的交接就是一个状态报告，而非交接。

### 交接是生成的，不是写的

生成器读工作台产物并产出包，是对工作台状态的总结，而不是去手写一份摘要。

### 两种形态：人读的和机器读的

`handoff.md`是给人读的，`handoff.json`是给机器读的，下一个agent负责加载。来自相同的物料，如果有分歧，以JSON为主。

### 反馈日志裁剪

完整的反馈日志很长，交接应该只包含最后几条，然后加上返回值异常的那些。下一个会话可以根据需要读取完整的反馈日志，但是交接的包需要保持小。

### 留一个干净状态

交接描述了工作，干净的状态让工作可续。这俩不是一个事。如果下一个会话打开时有应用了一半的diff，agent忘记的临时文件，错误的测试文件，再完美的`handoff.md`也没有意义。下一个agent就需要花费十分钟来清理烂摊子而不是直接上手，这可能发生在工作过程中的每次会话。

因此当功能可用的时候不代表会话结束了。只有当工作台处于一个生成器可以总结且下一个会话可以信任的状态时，才算结束。在交接发生之前。清理自成一个阶段。

检查项包括:
1. 工作树。
2. 临时物料。
3. 测试。
4. 功能看板。
5. 分支。

清理阶段应该吐出`clean_state.json`文件标识阻塞项目。空列表是生成器在写交接包之前的断言前置条件。一个建在脏树上的交接不是交接，相当于把烂摊子转发了出去。这两个产物要配对：清理证明工作台可以安全地让出，交接证明下个会话知道应该从哪里开始。

# 开始编码

对应本章核心：**交接包由产物生成（七字段，`next_action` 承重）**、**`handoff.md` + `handoff.json`（分歧以 JSON 为准）**、**反馈裁剪**、**先 `clean_state.json`（空 blockers）再生成**。  
玩具先跑通脏树拒绝与双形态包；生产段用 **LangChain 工具 + DeepSeek** 走清理→生成→下一会话加载。不硬凑 PyTorch。


## 1. 教学玩具：Cleaner + HandoffGenerator

- 五项清理：工作树 / 临时物料 / 测试 / 看板 / 分支。
- 生成器读 state / gate / review / feedback；无 `next_action` 或脏树直接拒绝。


In [ ]:
from __future__ import annotations

import json
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any


class DirtyWorkbench(RuntimeError):
    """脏树不能写交接：那只是在转发烂摊子。"""


class IncompleteHandoff(RuntimeError):
    """没有 next_action 的包只是状态报告，不是交接。"""


@dataclass
class CleanState:
    """清理阶段产物：空 blockers 才允许生成交接。"""

    worktree_clean: bool = True
    temp_cleared: bool = True
    tests_settled: bool = True
    board_consistent: bool = True
    branch_ok: bool = True
    blockers: list[str] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @property
    def ok(self) -> bool:
        return not self.blockers


@dataclass
class HandoffPack:
    """机器可读交接：七个承重字段。"""

    summary: str
    changed_files: list[str]
    command_runs: list[str]
    failed_attempts: list[dict[str, Any]]
    open_risks: list[str]
    next_action: str
    verdict_pointer: dict[str, str]

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def trim_feedback(rows: list[dict[str, Any]], *, tail: int = 3) -> list[dict[str, Any]]:
    """
    保留最后 tail 条，并并入所有异常退出。

    Args:
        rows: 完整反馈。
        tail: 尾部条数。
    Returns:
        trimmed: 去重后的短列表。
    """
    failed = [r for r in rows if r.get("exit_code") not in (0,)]
    last = rows[-tail:] if rows else []
    seen: set[str] = set()
    out: list[dict[str, Any]] = []
    for r in failed + last:
        key = json.dumps(r, ensure_ascii=False, sort_keys=True)
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out


class Cleaner:
    """会话结束前的清理阶段。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)
        self.path = self.root / "clean_state.json"

    def inspect(self) -> CleanState:
        """
        Returns:
            state: 五项检查 + blockers。
        """
        blockers: list[str] = []
        flags = {
            "worktree_clean": not (self.root / ".half_applied.diff").exists(),
            "temp_cleared": not any(self.root.glob("*.tmp")),
            "tests_settled": not (self.root / "tests" / "broken.flag").exists(),
            "board_consistent": True,
            "branch_ok": True,
        }
        board = {}
        bp = self.root / "task_board.json"
        if bp.exists():
            board = json.loads(bp.read_text(encoding="utf-8"))
            if board.get("stale"):
                flags["board_consistent"] = False
        branch = self.root / "branch.txt"
        if branch.exists() and branch.read_text(encoding="utf-8").strip() == "detached":
            flags["branch_ok"] = False
        labels = {
            "worktree_clean": "half-applied diff",
            "temp_cleared": "temp files",
            "tests_settled": "broken tests",
            "board_consistent": "stale board",
            "branch_ok": "bad branch",
        }
        for k, ok in flags.items():
            if not ok:
                blockers.append(labels[k])
        return CleanState(**flags, blockers=blockers)

    def cleanup(self) -> CleanState:
        """
        清掉半成品 diff / tmp / 坏测试标记。

        Returns:
            state: 清理后的检查结果。
        """
        half = self.root / ".half_applied.diff"
        if half.exists():
            half.unlink()
        for tmp in self.root.glob("*.tmp"):
            tmp.unlink()
        flag = self.root / "tests" / "broken.flag"
        if flag.exists():
            flag.unlink()
        bp = self.root / "task_board.json"
        if bp.exists():
            board = json.loads(bp.read_text(encoding="utf-8"))
            board["stale"] = False
            bp.write_text(json.dumps(board, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        br = self.root / "branch.txt"
        if br.exists() and br.read_text(encoding="utf-8").strip() == "detached":
            br.write_text("feat/login-validation\n", encoding="utf-8")
        state = self.inspect()
        self.path.write_text(json.dumps(state.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        return state


class HandoffGenerator:
    """读工作台产物生成交接；脏树拒绝。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)
        self.json_path = self.root / "handoff.json"
        self.md_path = self.root / "handoff.md"
        self.cleaner = Cleaner(self.root)

    def _load_json(self, name: str) -> dict[str, Any]:
        p = self.root / name
        if not p.exists():
            return {}
        return json.loads(p.read_text(encoding="utf-8"))

    def _load_jsonl(self, name: str) -> list[dict[str, Any]]:
        p = self.root / name
        if not p.exists():
            return []
        return [json.loads(ln) for ln in p.read_text(encoding="utf-8").splitlines() if ln.strip()]

    def generate(self, *, require_clean: bool = True) -> HandoffPack:
        """
        Args:
            require_clean: 要求 clean_state.blockers 为空。
        Returns:
            pack: 已落盘的交接包。
        """
        if require_clean:
            clean = self.cleaner.inspect()
            if not clean.ok:
                raise DirtyWorkbench(f"refuse handoff on dirty tree: {clean.blockers}")
            if not self.cleaner.path.exists():
                self.cleaner.path.write_text(
                    json.dumps(clean.to_dict(), ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8",
                )

        state = self._load_json("agent_state.json")
        gate = self._load_json("verification_report.json")
        review = self._load_json("review_report.json")
        feedback = self._load_jsonl("feedback_record.jsonl")
        changed = list(state.get("touched_files") or self._load_json("changed_files.json") or [])
        if isinstance(changed, dict):
            changed = list(changed.get("files", []))

        next_action = str(state.get("next_action") or "").strip()
        if not next_action:
            raise IncompleteHandoff("next_action is required; otherwise this is a status report")

        trimmed = trim_feedback(feedback)
        runs = []
        for r in trimmed:
            cmd = r.get("command", [])
            runs.append(" ".join(cmd) if isinstance(cmd, list) else str(cmd))
        failed = [
            {
                "command": r.get("command"),
                "exit_code": r.get("exit_code"),
                "why": r.get("agent_note") or r.get("error") or "nonzero/null exit",
            }
            for r in trimmed
            if r.get("exit_code") != 0
        ]
        risks = list(state.get("blockers") or [])
        if review.get("verdict") in {"soft_fail", "hard_fail"}:
            risks.append(f"review {review.get('verdict')}")

        pack = HandoffPack(
            summary=str(state.get("progress_note") or state.get("summary") or "session progress"),
            changed_files=[str(x) for x in changed],
            command_runs=runs,
            failed_attempts=failed,
            open_risks=risks,
            next_action=next_action,
            verdict_pointer={
                "verification": "verification_report.json",
                "review": "review_report.json",
            },
        )
        self.json_path.write_text(json.dumps(pack.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        self.md_path.write_text(self._to_markdown(pack, gate, review), encoding="utf-8")
        return pack

    @staticmethod
    def _to_markdown(pack: HandoffPack, gate: dict[str, Any], review: dict[str, Any]) -> str:
        def bullets(items: list[str]) -> list[str]:
            return items or ["- (none)"]

        lines = [
            "# Handoff",
            "",
            "## Summary",
            pack.summary,
            "",
            "## Changed files",
            ", ".join(pack.changed_files) or "(none)",
            "",
            "## Commands",
            *bullets([f"- `{c}`" for c in pack.command_runs]),
            "",
            "## Failed attempts",
            *bullets([f"- {json.dumps(f, ensure_ascii=False)}" for f in pack.failed_attempts]),
            "",
            "## Open risks",
            *bullets([f"- {r}" for r in pack.open_risks]),
            "",
            "## Next action",
            pack.next_action,
            "",
            "## Verdicts",
            f"- verification: `{pack.verdict_pointer['verification']}` passed={gate.get('passed')}",
            f"- review: `{pack.verdict_pointer['review']}` verdict={review.get('verdict')}",
            "",
            "_Machine source of truth is handoff.json._",
            "",
        ]
        return "\n".join(lines)

    def load_for_next_session(self) -> dict[str, Any]:
        """
        Returns:
            pack: 下一会话加载的 JSON（与 md 分歧时以 JSON 为准）。
        """
        if not self.json_path.exists():
            raise FileNotFoundError("handoff.json missing")
        data = json.loads(self.json_path.read_text(encoding="utf-8"))
        if not str(data.get("next_action") or "").strip():
            raise IncompleteHandoff("handoff.json missing next_action")
        return data


def write_session_artifacts(root: Path, *, dirty: bool = False, missing_next: bool = False) -> Path:
    """
    写入上一会话物料。

    Args:
        root: 仓库根。
        dirty: 留下半成品 / tmp。
        missing_next: 故意不写 next_action。
    Returns:
        root: 同一目录。
    """
    root.mkdir(parents=True, exist_ok=True)
    (root / "tests").mkdir(exist_ok=True)
    state = {
        "active_task_id": "T-LOGIN",
        "touched_files": ["src/auth/login.py"],
        "progress_note": "补了空密码拒绝，pytest 曾红过一次后转绿",
        "blockers": ["staging 权限未批"],
        "next_action": "" if missing_next else "用 staging token 再跑一次验收",
        "assumptions": ["登陆接口已有骨架"],
    }
    (root / "agent_state.json").write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    (root / "verification_report.json").write_text(
        json.dumps({"passed": True, "task_id": "T-LOGIN"}, ensure_ascii=False),
        encoding="utf-8",
    )
    (root / "review_report.json").write_text(
        json.dumps({"verdict": "soft_fail", "total": 6}, ensure_ascii=False),
        encoding="utf-8",
    )
    rows = [
        {"id": "1", "command": ["pytest", "-q"], "exit_code": 1, "agent_note": "空密码用例未写"},
        {"id": "2", "command": ["echo", "noise"], "exit_code": 0, "agent_note": "中间命令"},
        {"id": "3", "command": ["ls"], "exit_code": 0, "agent_note": "探路"},
        {"id": "4", "command": ["pytest", "-q"], "exit_code": 0, "agent_note": "补测后转绿"},
    ]
    (root / "feedback_record.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows),
        encoding="utf-8",
    )
    (root / "task_board.json").write_text(json.dumps({"stale": dirty, "tasks": []}, ensure_ascii=False), encoding="utf-8")
    (root / "branch.txt").write_text("feat/login-validation\n", encoding="utf-8")
    if dirty:
        (root / ".half_applied.diff").write_text("+++ leftover\n", encoding="utf-8")
        (root / "scratch.tmp").write_text("tmp\n", encoding="utf-8")
        (root / "tests" / "broken.flag").write_text("1\n", encoding="utf-8")
    return root


print("handoff ready | clean_state + generated md/json pack")


## 2. 玩具示例

脏树拒绝、清理后生成、JSON 压过被改写的 md、缺 next_action 拒绝。


In [ ]:
def demo_handoff() -> None:
    """脏树拒绝、缺 next_action 拒绝、清理后生成、下一会话只读 JSON。"""
    root = Path(tempfile.mkdtemp(prefix="handoff_"))
    try:
        write_session_artifacts(root, dirty=True)
        gen = HandoffGenerator(root)
        try:
            gen.generate()
            raise AssertionError("dirty must refuse")
        except DirtyWorkbench:
            print("dirty tree refuse ok")

        clean = gen.cleaner.cleanup()
        assert clean.ok and not clean.blockers
        assert gen.cleaner.path.exists()
        print("cleanup emptied blockers ok")

        pack = gen.generate()
        assert pack.next_action.startswith("用 staging")
        assert "src/auth/login.py" in pack.changed_files
        assert any("pytest" in c for c in pack.command_runs)
        assert any(f.get("exit_code") == 1 for f in pack.failed_attempts)
        assert gen.json_path.exists() and gen.md_path.exists()
        # 反馈被裁剪：完整 4 条，交接不应原样塞满 noise+ls 除非在 tail/fail 里
        assert len(pack.command_runs) <= 4
        print("generated md+json ok")

        nxt = gen.load_for_next_session()
        assert nxt["next_action"] == pack.next_action
        # 人为改 md 不应影响机器源
        gen.md_path.write_text("# stale human edit\n", encoding="utf-8")
        nxt2 = gen.load_for_next_session()
        assert nxt2["next_action"] == pack.next_action
        print("json wins over md ok")

        write_session_artifacts(root, dirty=False, missing_next=True)
        gen.cleaner.cleanup()
        try:
            HandoffGenerator(root).generate()
            raise AssertionError("missing next_action")
        except IncompleteHandoff:
            print("no next_action refuse ok")

        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_handoff()


## 3. 生产级：LangChain 交接工具 + DeepSeek

工具：`run_cleanup` / `generate_handoff` / `load_handoff`。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
import tempfile
import threading
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_GEN: HandoffGenerator | None = None
PROD_LOCK = threading.Lock()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_handoff_repo(*, dirty: bool = True) -> str:
    """
    Args:
        dirty: 是否从脏树开始。
    Returns:
        json: root。
    """
    global PROD_ROOT, PROD_GEN
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = Path(tempfile.mkdtemp(prefix="handoff_prod_"))
    write_session_artifacts(PROD_ROOT, dirty=dirty)
    PROD_GEN = HandoffGenerator(PROD_ROOT)
    return json.dumps({"root": str(PROD_ROOT), "dirty": dirty}, ensure_ascii=False)


def _gen() -> HandoffGenerator:
    if PROD_GEN is None:
        reset_handoff_repo(dirty=True)
    assert PROD_GEN is not None
    return PROD_GEN


class EmptyArgs(BaseModel):
    pass


class ResetArgs(BaseModel):
    dirty: bool = True


def build_handoff_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        with PROD_LOCK:
            return reset_handoff_repo(dirty=ResetArgs(**kwargs).dirty)

    def _inspect(**kwargs: Any) -> str:
        with PROD_LOCK:
            st = _gen().cleaner.inspect()
            return json.dumps(st.to_dict(), ensure_ascii=False)

    def _cleanup(**kwargs: Any) -> str:
        with PROD_LOCK:
            st = _gen().cleaner.cleanup()
            return json.dumps(st.to_dict(), ensure_ascii=False)

    def _generate(**kwargs: Any) -> str:
        with PROD_LOCK:
            try:
                pack = _gen().generate(require_clean=True)
                return json.dumps({"ok": True, "pack": pack.to_dict()}, ensure_ascii=False)
            except (DirtyWorkbench, IncompleteHandoff) as e:
                return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)

    def _load(**kwargs: Any) -> str:
        with PROD_LOCK:
            try:
                data = _gen().load_for_next_session()
                return json.dumps({"ok": True, "handoff": data, "source": "handoff.json"}, ensure_ascii=False)
            except (FileNotFoundError, IncompleteHandoff) as e:
                return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)

    def _read_md(**kwargs: Any) -> str:
        with PROD_LOCK:
            p = _gen().md_path
            if not p.exists():
                return json.dumps({"ok": False, "error": "handoff.md missing"}, ensure_ascii=False)
            return p.read_text(encoding="utf-8")

    return [
        StructuredTool.from_function(name="reset_repo", description="重建上一会话物料。", func=_reset, args_schema=ResetArgs),
        StructuredTool.from_function(name="inspect_clean_state", description="检查五项清理项。", func=_inspect, args_schema=EmptyArgs),
        StructuredTool.from_function(name="run_cleanup", description="清理后写 clean_state.json。", func=_cleanup, args_schema=EmptyArgs),
        StructuredTool.from_function(
            name="generate_handoff",
            description="从产物生成 handoff.md + handoff.json；脏树或无 next_action 会失败。",
            func=_generate,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(
            name="load_handoff",
            description="下一会话加载 handoff.json（机器源）。",
            func=_load,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(name="read_handoff_md", description="读给人看的 handoff.md。", func=_read_md, args_schema=EmptyArgs),
    ]


HANDOFF_TOOLS = build_handoff_tools()


def build_handoff_agent():
    """
    Returns:
        agent: 先清理再生成交接的 control agent。
    """
    system = (
        "你是多会话交接代理。会话结束前必须先 run_cleanup，"
        "clean_state.blockers 为空才能 generate_handoff。\n"
        "下一会话只 load_handoff（JSON），不要靠聊天回忆。\n"
        "脏树不要硬写交接。用中文简短说明。"
    )
    return create_agent(get_llm(), HANDOFF_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 28) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:280]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"handoff production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍验证脏树拒绝与下一会话加载。


In [ ]:
def demo_prod_handoff() -> None:
    """脚本化清理门禁 +（有 key 时）DeepSeek 交接。"""
    reset_handoff_repo(dirty=True)
    try:
        _gen().generate()
        raise AssertionError("dirty should fail")
    except DirtyWorkbench:
        pass
    _gen().cleaner.cleanup()
    pack = _gen().generate()
    nxt = _gen().load_for_next_session()
    assert nxt["next_action"] == pack.next_action
    print("=== scripted ===")
    print(json.dumps({"dirty_refused": True, "cleaned": True, "next": nxt["next_action"]}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_handoff_agent()
    prompt = (
        "reset_repo dirty=true，先 generate_handoff（应失败），"
        "inspect_clean_state，run_cleanup，再 generate_handoff，"
        "load_handoff，read_handoff_md，中文说明为何必须先清理再交接。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    loaded = _gen().load_for_next_session()
    assert loaded["next_action"]
    assert _gen().json_path.exists() and _gen().md_path.exists()
    clean = json.loads(_gen().cleaner.path.read_text(encoding="utf-8"))
    assert clean.get("blockers") == []
    print("PROD DEMO OK")


demo_prod_handoff()
